# Model 3: RAG Pipeline for Austrian Tax Law Q&A

**Author:** Berkay Kaya (Team 11)  
**Course:** Applications of Data Science -- SBWL Data Science, WU Wien, SS26

## Overview
This notebook implements a retrieval-augmented generation (RAG) pipeline for answering Austrian tax law questions in German. Instead of relying only on the parametric knowledge of the language model, the system first retrieves relevant passages from Austrian tax law documents and then uses them as context for answer generation. The goal is to improve factual grounding, produce more precise legal references, and reduce hallucinations compared to a plain inference-only setup.

## Architecture
1. **PDF Ingestion:** Extract text from Austrian tax law PDFs (EStG, KStG, UStG) using PyMuPDF  
2. **Chunking:** Split on paragraph markers to create legally meaningful chunks  
3. **Retrieval:** Encode chunks with `paraphrase-multilingual-mpnet-base-v2`, index with FAISS, retrieve top-k  
4. **Generation:** Feed retrieved context + question to `google/gemma-2-2b-it` (4-bit quantized) to produce answers  

## Requirements
- Kaggle notebook with **GPU T4 x2** and **Internet** enabled  
- Input dataset: `austrian-tax-laws` (contains `EStG.pdf`, `KStG.pdf`, `UStG.pdf`, `dataset_clean.csv`)  

## Cell 1 — Install Packages
Required Python packages for the RAG notebook.

- `!{sys.executable} -m pip install ...` runs a package installation command inside the same Python environment used by the notebook.
- `transformers` is the Hugging Face library for loading pretrained language models and tokenizers.
- `accelerate` helps manage efficient model execution on available hardware.
- `bitsandbytes` enables 4-bit quantization to reduce memory usage during model loading.
- `sentence-transformers` provides embedding models that convert text into semantic vectors.
- `faiss-cpu` is Facebook AI Similarity Search for fast vector retrieval on the CPU.
- `PyMuPDF` is used to read legal PDF files and is later imported as `fitz`.
- `pandas` is used for reading datasets and writing final CSV outputs.
- `-q` means “quiet” and suppresses long installation logs.

Compared with model 2, the important additions are `sentence-transformers` and `faiss-cpu`, because RAG needs both semantic embeddings and a retrieval index.
The CPU version of FAISS is fully sufficient here because the project works with only a few thousand legal text chunks.

### Project Impact
This cell prepares the technical basis for the retrieval pipeline. Without these packages, the notebook could neither build the search index nor generate answers grounded in the Austrian tax-law source texts.


In [ ]:
import sys
!{sys.executable} -m pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu PyMuPDF pandas

# faiss-cpu is used instead of faiss-gpu — sufficient for our dataset size (643 questions)

## Cell 2 — Imports
Additional libraries needed specifically for retrieval-augmented generation.

- `import numpy as np` loads NumPy, which is used for efficient operations on arrays and vectors.
- `import faiss` imports the FAISS library for similarity search over embedding vectors.
- `from sentence_transformers import SentenceTransformer` imports the embedding model class that converts text into semantic vector representations.

These imports are new compared with model 2 because model 3 does not rely only on generation.
It first converts law chunks and user questions into vectors, then retrieves the most relevant chunks before generating an answer.

### Project Impact
This cell introduces the core retrieval tools of the RAG system. Without these imports, the project could not search the legal corpus semantically and would lose its main advantage over plain prompting.

In [10]:
import os
import re
import json
import time
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 3 — Configuration
Defines the main configuration values for the RAG pipeline.

- `RETRIEVER_MODEL` specifies the embedding model used for semantic search.
- `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` is multilingual, which is important because the Austrian tax-law questions and sources are in German.
- `GENERATOR_MODEL` defines the language model that produces the final answer.
- `google/gemma-2-2b-it` is the instruction-tuned Gemma model used for answer generation.
- `TOP_K = 5` means the retriever returns the five most relevant law chunks for each question.
- `CHUNK_MAX_LEN = 2000` limits the maximum size of each text chunk in characters.

The retriever and generator have different roles in RAG: one searches, the other writes the answer. `TOP_K` and `CHUNK_MAX_LEN` directly control how much legal context is available during generation.

### Project Impact
This cell sets the most important retrieval and generation choices in the project. These values strongly influence answer quality, because they determine which chunks are retrieved and how much source material reaches the model.


In [11]:
# --- Paths (Kaggle) ---
PDF_DIR = "/kaggle/input/datasets/berkaya462/a-o-ds-4/"
DATASET_PATH = "/kaggle/input/datasets/berkaya462/a-o-ds-4/dataset_clean.csv"
OUTPUT_CSV = "/kaggle/working/model3_rag_results.csv"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- PDF files to process ---
PDF_FILES = {
    "EStG": os.path.join(PDF_DIR, "EStG.pdf"),
    "KStG": os.path.join(PDF_DIR, "KStG 1988 Fassung vom 03.04.2026.pdf"),
    "UStG": os.path.join(PDF_DIR, "UStG.pdf"),
}

# --- Model names ---
RETRIEVER_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
GENERATOR_MODEL = "google/gemma-2-2b-it"

# --- Retrieval settings ---
TOP_K = 5            # number of chunks to retrieve per question
CHUNK_MAX_LEN = 2000 # max characters per chunk

# --- Generation settings ---
MAX_NEW_TOKENS = 300   # 300 is enough for concise legal answers (same as Model 1)
REPETITION_PENALTY = 1.15

print("Configuration loaded.")
for name, path in PDF_FILES.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗ NOT FOUND'} — {path}")

Configuration loaded.
  EStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/EStG.pdf
  KStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/KStG 1988 Fassung vom 03.04.2026.pdf
  UStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/UStG.pdf


## Cell 4 — Read PDF Files
A function that reads legal source documents from PDF files.

- `def extract_pdf_text(pdf_path):` creates a reusable function for extracting text from one PDF file.
- `fitz.open(pdf_path)` opens the PDF document with PyMuPDF.
- `text = ""` initializes an empty string that will collect the extracted text.
- `for page in doc:` loops through the PDF page by page.
- `page.get_text()` extracts the plain text content of each page.
- `text += ...` appends each page’s text to the full document string.
- `doc.close()` closes the PDF file and frees memory.
- `return text` returns the final extracted text.

This is the first step of building the RAG knowledge base, because the legal documents must be converted from PDF into raw text before they can be split, embedded, and retrieved.

### Project Impact
This cell turns the original legal PDFs into machine-readable text. If the project cannot extract the source material correctly, the retrieval stage will have no reliable knowledge base to search over.


In [12]:
def extract_pdf_text(pdf_path):
    """Extract all text from a PDF file using PyMuPDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

# Extract text from all three PDFs
law_texts = {}
for law_name, pdf_path in PDF_FILES.items():
    law_texts[law_name] = extract_pdf_text(pdf_path)
    print(f"{law_name}: {len(law_texts[law_name]):,} characters extracted")

print(f"\nTotal laws loaded: {len(law_texts)}")

EStG: 948,838 characters extracted
KStG: 214,212 characters extracted
UStG: 312,566 characters extracted

Total laws loaded: 3


## Cell 5 — Split Legal Text into Chunks
Splitting the extracted legal text into structured chunks that can later be embedded and retrieved.

- `current_section = "Preamble"` stores a default section name before the first law paragraph appears.
- `current_text = ""` starts an empty buffer for the text of the current section.
- `for part in parts:` loops through the previously separated text fragments.
- `re.match(r'§\s*\d+[a-z]*', part)` checks whether the current fragment begins with a paragraph marker such as `§ 7` or `§ 7a`.
- `if current_text.strip():` ensures that the previous section is only saved if it actually contains text.
- `chunks.append({...})` stores one chunk together with metadata such as the law name and section identifier.
- `current_section = part.strip()` updates the current section label when a new paragraph begins.
- `current_text = part` starts collecting the new section.
- `current_text += part` appends normal text to the current section if no new paragraph marker is found.
- `for i in range(0, len(txt), max_len):` splits very long sections into smaller pieces.
- `piece = txt[i:i + max_len]` extracts one sub-chunk with a maximum character length.

This chunking logic is more retrieval-oriented. It preserves the law name and section reference, which later helps the retriever return context that is both semantically relevant and easy to cite.

### Project Impact
This cell transforms long legal documents into searchable units. Good chunking is crucial because retrieval quality depends directly on whether the law text is broken into useful, well-labeled sections.


In [13]:
def chunk_law_text(text, law_name, max_len=CHUNK_MAX_LEN):
    """Split law text on § markers and return list of chunk dicts."""
    # Split on § but keep the delimiter
    parts = re.split(r'(§\s*\d+[a-z]*)', text)

    chunks = []
    current_section = "Preamble"
    current_text = ""

    for part in parts:
        # Check if this part is a section marker like "§ 4" or "§ 10a"
        if re.match(r'§\s*\d+[a-z]*', part):
            # Save previous chunk if it has content
            if current_text.strip():
                chunks.append({
                    "law": law_name,
                    "section": current_section,
                    "text": current_text.strip()
                })
            current_section = part.strip()
            current_text = part  # start new chunk with the § marker
        else:
            current_text += part

    # Don't forget the last chunk
    if current_text.strip():
        chunks.append({
            "law": law_name,
            "section": current_section,
            "text": current_text.strip()
        })

    # Split chunks that are too long
    final_chunks = []
    for chunk in chunks:
        txt = chunk["text"]
        if len(txt) <= max_len:
            final_chunks.append(chunk)
        else:
            # Split long chunks into pieces of max_len characters
            for i in range(0, len(txt), max_len):
                piece = txt[i:i + max_len]
                final_chunks.append({
                    "law": chunk["law"],
                    "section": chunk["section"],
                    "text": piece.strip()
                })

    return final_chunks

# Chunk all laws
all_chunks = []
for law_name, text in law_texts.items():
    law_chunks = chunk_law_text(text, law_name)
    all_chunks.extend(law_chunks)
    print(f"{law_name}: {len(law_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Show a sample chunk
print(f"\nSample chunk:")
print(f"  Law: {all_chunks[5]['law']}")
print(f"  Section: {all_chunks[5]['section']}")
print(f"  Text (first 200 chars): {all_chunks[5]['text'][:200]}...")

EStG: 3788 chunks
KStG: 766 chunks
UStG: 904 chunks

Total chunks: 5458

Sample chunk:
  Law: EStG
  Section: Preamble
  Text (first 200 chars): 10 (NR: GP XXIV RV 981 AB 1026 S. 90. BR: 8437 AB 8439 S. 792.) 
[CELEX-Nr.: 32010L0012] 
BGBl. I Nr. 41/2011 (VfGH) 
BGBl. I Nr. 76/2011 (NR: GP XXIV RV 1212 AB 1320 S. 114. BR: 8524 AB 8558 S. 799.)...


## Cell 6 — Build the FAISS Index
Creates the vector search index that forms the core of the RAG system.

- `SentenceTransformer(RETRIEVER_MODEL)` loads the embedding model used for semantic retrieval.
- `retriever_model.encode(...)` converts all law chunks into vector representations.
- `chunk_texts` is the list of legal text chunks that will be embedded.
- `batch_size=64` processes multiple chunks at once for better speed.
- `convert_to_numpy=True` returns NumPy arrays, which FAISS expects.
- `normalize_embeddings=True` scales each vector to unit length, so inner product becomes equivalent to cosine similarity.
- `show_progress_bar=True` displays embedding progress while the chunks are being encoded.
- `device="cuda" if torch.cuda.is_available() else "cpu"` uses a GPU if available, otherwise the CPU.
- `embeddings.shape[1]` gives the vector dimension.
- `faiss.IndexFlatIP(dim)` creates a FAISS index based on inner product similarity.
- `index.add(embeddings)` adds all chunk vectors to the search index.

In simple terms, this cell converts all relevant law passages into numeric vectors and places them in a fast similarity-search structure. Later, the same process is applied to a question, and the nearest chunk vectors are retrieved as context.

### Project Impact
This cell builds the actual retrieval engine of the project. It is what allows the notebook to search the law corpus semantically instead of relying only on what the language model remembers.


In [14]:
# Load the retriever model
print("Loading SentenceTransformer retriever model...")
retriever_model = SentenceTransformer(RETRIEVER_MODEL)
print(f"Model loaded: {RETRIEVER_MODEL}")

# Encode all chunk texts
chunk_texts = [c["text"] for c in all_chunks]

print(f"Encoding {len(chunk_texts)} chunks...")
start = time.time()
embeddings = retriever_model.encode(
    chunk_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,  # needed for inner-product similarity
    show_progress_bar=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Encoding done in {time.time() - start:.1f}s, shape: {embeddings.shape}")

# Build FAISS index (inner product = cosine similarity on normalized vectors)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors (dim={dim})")

Loading SentenceTransformer retriever model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Encoding 5458 chunks...


Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Encoding done in 17.2s, shape: (5458, 768)
FAISS index built with 5458 vectors (dim=768)


## Cell 7 — Keyword Pre-Filtering and Retrieval
Adds a keyword-based law filter on top of semantic retrieval.

- `LAW_KEYWORDS` defines simple keyword lists for major Austrian tax laws.
- `detect_law(question)` tries to infer which law is most likely relevant to the question.
- `question.lower()` converts the question to lowercase so matching works regardless of capitalization.
- `any(kw in q_lower for kw in keywords)` checks whether at least one law-specific keyword appears in the question.
- `return law` returns the detected law abbreviation, such as `KStG`, `UStG`, or `EStG`.
- `q_emb = retriever_model.encode(...)` embeds the user question into a vector.
- `index.search(q_emb, top_k * 3)` retrieves more than the final number of chunks to allow later filtering.
- `scores` are similarity values and `indices` are the positions of retrieved chunks in the FAISS index.
- `matching` stores chunks from the detected law.
- `other` stores remaining semantically similar chunks from different laws.
- `results = matching[:top_k]` prioritizes chunks from the most likely law first.
- `results.extend(other[:remaining])` fills empty slots with the best remaining chunks if needed.

This hybrid design improves robustness. Pure semantic retrieval sometimes returns paragraphs from the wrong law because tax questions across different statutes can use similar language.

### Project Impact
This cell improves retrieval precision before answer generation even begins. It reduces the risk that the model receives legally similar but wrong context, which is especially important in a tax-law project.


In [15]:
# Keywords to detect which law a question is about
LAW_KEYWORDS = {
    "KStG": ["körperschaft", "kstg", "köst"],
    "UStG": ["umsatzsteuer", "ustg", "vorsteuer", "mehrwertsteuer"],
    "EStG": ["einkommensteuer", "estg", "lohnsteuer"],
}

def detect_law(question):
    """Detect which law a question is primarily about."""
    q_lower = question.lower()
    for law, keywords in LAW_KEYWORDS.items():
        if any(kw in q_lower for kw in keywords):
            return law
    return None  # no specific law detected

def retrieve(question, top_k=TOP_K):
    """Retrieve top-k relevant chunks, prioritizing the detected law."""
    law_filter = detect_law(question)

    # Encode the question
    q_emb = retriever_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    # Search more candidates than needed, then filter
    scores, indices = index.search(q_emb, top_k * 3)

    # Separate into matching law and other
    matching = []
    other = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = all_chunks[idx]
        entry = {
            "law": chunk["law"],
            "section": chunk["section"],
            "text": chunk["text"],
            "score": float(score)
        }
        if law_filter and chunk["law"] == law_filter:
            matching.append(entry)
        else:
            other.append(entry)

    # Prioritize matching law chunks, fill remaining with best others
    results = matching[:top_k]
    remaining = top_k - len(results)
    if remaining > 0:
        results.extend(other[:remaining])

    return results

# Quick test
test_results = retrieve("Was ist die Bemessungsgrundlage der Körperschaftsteuer?")
for r in test_results:
    print(f"  [{r['law']} {r['section']}] score={r['score']:.3f} | {r['text'][:80]}...")

  [KStG § 27] score=0.813 | § 27 
Abs. 5 
Z 7 
des 
Einkommensteuergesetzes 1988, für die Kapitalertragsteue...
  [KStG § 7] score=0.767 | § 7. (1) Der Körperschaftsteuer ist das Einkommen zugrunde zu legen, das der unb...
  [KStG § 6] score=0.761 | § 6 Z 14 lit. b des Einkommensteuergesetzes 1988 ist sinngemäß 
anzuwenden. Bei ...
  [KStG § 22] score=0.752 | § 22. (1) Die Körperschaftsteuer vom Einkommen (...
  [KStG § 2] score=0.745 | § 2 
Abs. 8 
Z 4 
des 
Einkommensteuergesetzes 1988 nachzuversteuern sind, 
 
– ...


## Cell 8+9 — Load the Generator Model
Prepares the generator model for inference.

- `model.eval()` switches the model into evaluation mode.
- In evaluation mode, layers such as dropout are disabled.
- Dropout randomly deactivates neurons during training, but this behavior is not wanted when generating final answers.
- The remaining setup, such as 4-bit quantization, tokenizer loading, and `BitsAndBytesConfig`, is the same as in model 2.

The generator is still Gemma, but in model 3 it is no longer asked to answer from memory alone. Instead, it works together with the retriever and generates an answer based on the retrieved legal context.

### Project Impact
This cell ensures stable and deterministic inference behavior for the final RAG answers. It helps the project produce more consistent outputs during evaluation and submission generation.


In [16]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load HF token from Kaggle Secrets and log in
secret = UserSecretsClient().get_secret("HF_TOKEN")
login(token=secret)
print("HuggingFace login successful.")

HuggingFace login successful.


In [17]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print(f"Loading tokenizer for {GENERATOR_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
print(f"Loading model {GENERATOR_MODEL} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Generator model loaded successfully.")

Loading tokenizer for google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Generator model loaded successfully.


## Cell 10 — RAG Prompt and Generation
Building the final RAG prompt and prepares the model input for generation.

- `PROMPT_TEMPLATE` defines the structure of the instruction sent to the language model.
- `{context}` is filled with the retrieved law chunks.
- `{question}` is filled with the user’s actual tax-law question.
- `format_context(chunks)` converts the retrieved chunks into a readable text block for the model.
- `f"[{c['law']} {c['section']}]\n{c['text']}"` preserves source metadata and chunk text together.
- `"\n\n".join(parts)` combines all retrieved chunks into one formatted context string.
- `generate_answer(...)` is the main function for RAG answer generation.
- `chunks = retrieve(question, ...)` performs retrieval first.
- `context = format_context(chunks)` formats the retrieved chunks.
- `prompt = PROMPT_TEMPLATE.format(...)` inserts context and question into the prompt template.
- `tokenizer.apply_chat_template(...)` applies Gemma’s built-in chat format automatically.
- `add_generation_prompt=True` appends the model-response marker so generation can begin correctly.
- `tokenizer(..., return_tensors="pt")` converts the prompt into PyTorch tensors.
- `truncation=True, max_length=4096` ensures the prompt stays within the model’s context window.
- `.to(model.device)` moves the input tensors to the same device as the model.

This is the key difference between a normal generator and a RAG generator: the model is explicitly shown relevant legal passages before it answers.

### Project Impact
This cell is where retrieval and generation are combined into one workflow. It gives the project grounded, source-aware answers instead of relying only on the model’s internal memory.


In [18]:
PROMPT_TEMPLATE = """Du bist ein Experte für österreichisches Steuerrecht.

Relevante Gesetzestexte:
{context}

Frage: {question}

Beantworte die Frage präzise und kurz auf Deutsch.
Zitiere die relevanten Paragraphen (z.B. § 7 Abs. 1 KStG).
Fokussiere dich auf die wichtigsten Rechtsnormen."""


def format_context(chunks):
    """Format retrieved chunks into a single context string."""
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(f"[{c['law']} {c['section']}]\n{c['text']}")
    return "\n\n".join(parts)


def generate_answer(question, top_k=TOP_K):
    """Full RAG pipeline: retrieve relevant chunks, build prompt, generate answer."""
    # Step 1: Retrieve relevant chunks
    chunks = retrieve(question, top_k=top_k)

    # Step 2: Build prompt
    context = format_context(chunks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    # Step 3: Tokenize (Gemma uses chat template)
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=4096).to(model.device)

    # Step 4: Generate — greedy decoding (faster + deterministic for factual answers)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Step 5: Decode only the new tokens (skip the input)
    input_len = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    return answer


# Quick test
test_answer = generate_answer("Was ist die Bemessungsgrundlage der Körperschaftsteuer?")
print("Test answer:")
print(test_answer)

Test answer:
Die Bemessungsgrundlage für die Körperschaftsteuer ist das **Einkommen**, welches der unbeschränkt Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat.  Dies wird gemäß § 7 Abs. 1 KStG bestimmt. 


**Wichtiger Hinweis:** Diese Antwort basiert auf den gegebenen Textfragmenten und kann keine umfassende rechtliche Beratung ersetzen.


## Cell 11 — Retrieval Validation
Performing a manual quality check of the retrieval step on the first validation questions.

- `for i in range(N_VALIDATE):` loops through a predefined number of validation examples.
- `chunks = retrieve(question)` retrieves the most relevant law chunks for the current question.
- `answer = generate_answer(question)` generates a full RAG answer using the retrieved context.
- `print("Retrieved:")` labels the retrieval output in the notebook.
- `print(f"  [{c['law']} {c['section']}] score={c['score']:.3f}")` shows which chunks were retrieved and how similar they are to the question.
- `score` is the similarity score between the question embedding and the chunk embedding.

This step is mainly for human inspection. It helps verify whether the retriever is actually finding the right laws and sections before running the full inference pipeline.

### Project Impact
This cell acts as an early quality-control step for the retrieval system. It can reveal problems in chunking, embedding, or law filtering before the project spends time generating all final answers.


In [19]:
# Load the dataset
df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded: {len(df)} questions")
print(f"Columns: {list(df.columns)}")
print(df.head())

# Validate on first 20 questions
N_VALIDATE = 20
print(f"\n{'='*80}")
print(f"VALIDATION: Running RAG on first {N_VALIDATE} questions")
print(f"{'='*80}\n")

for i in range(N_VALIDATE):
    row = df.iloc[i]
    qid = row["id"]
    question = row["prompt"]

    # Retrieve chunks
    chunks = retrieve(question)

    # Generate answer
    answer = generate_answer(question)

    # Print for inspection
    print(f"--- [{i+1}/{N_VALIDATE}] {qid} ---")
    print(f"Q: {question}")
    print(f"Retrieved:")
    for c in chunks:
        print(f"  [{c['law']} {c['section']}] score={c['score']:.3f}")
    print(f"A: {answer[:300]}{'...' if len(answer)>300 else ''}")
    print()

Dataset loaded: 643 questions
Columns: ['id', 'prompt']
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...
3  CORP-TAX-004  Wie wird eine Dividende einer österreichischen...
4  CORP-TAX-005  Was unterscheidet die steuerliche Behandlung e...

VALIDATION: Running RAG on first 20 questions

--- [1/20] CORP-TAX-001 ---
Q: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
Retrieved:
  [KStG § 27] score=0.806
  [KStG § 7] score=0.795
  [KStG § 6] score=0.781
  [KStG § 22] score=0.774
  [KStG § 17] score=0.767
A: Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist **das Einkommen**, welches der unbeschränkt Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat.  Dies wird nach § 7 Abs. 1 KStG bestimmt. 


**Wichtiger Hinweis:** Diese Ant

## Cell 12 — Inference Loop
Running the full question-answering loop for the RAG model and saves progress during execution.

- `checkpoint_file` stores the path to the checkpoint JSON file.
- `os.path.exists(checkpoint_file)` checks whether a checkpoint file already exists.
- `json.load(f)` reads previously saved results from the checkpoint file.
- `start_idx = len(results)` determines how many questions have already been processed.
- `time.time()` returns the current time in seconds.
- `elapsed = time.time() - start_time` calculates how long the current run has been going.
- `rate = (i + 1 - start_idx) / elapsed * 60` estimates how many questions are processed per minute.
- `json.dump(results, f, ensure_ascii=False)` writes the current results back to disk as JSON.
- `ensure_ascii=False` keeps German characters such as `ä`, `ö`, and `ü` readable in the file.
- If an error occurs during generation, the answer is set to `"Fehler bei der Generierung."` so the loop continues.

### Project Impact
This cell executes the full RAG pipeline across the dataset and protects progress during long runs. It is essential for reliability, especially because RAG inference is slower and more expensive than plain prompting.

In [20]:
# Check if a checkpoint exists (to resume from a crash)
checkpoint_file = os.path.join(CHECKPOINT_DIR, "rag_checkpoint.json")
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "r") as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Resuming from checkpoint: {start_idx} questions already done.")
else:
    results = []
    start_idx = 0
    print("Starting fresh inference.")

total = len(df)
print(f"Total questions: {total}")
print(f"Remaining: {total - start_idx}")
print()

start_time = time.time()

for i in range(start_idx, total):
    row = df.iloc[i]
    qid = row["id"]
    question = row["prompt"]

    try:
        answer = generate_answer(question)
    except Exception as e:
        print(f"  ERROR at {qid}: {e}")
        answer = "Fehler bei der Generierung."

    results.append({"id": qid, "answer": answer})

    # Progress log every 10 questions
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1 - start_idx) / elapsed * 60  # questions per minute
        print(f"  [{i+1}/{total}] {rate:.1f} q/min | last: {qid}")

    # Checkpoint every 100 questions
    if (i + 1) % 100 == 0:
        with open(checkpoint_file, "w") as f:
            json.dump(results, f, ensure_ascii=False)
        print(f"  >> Checkpoint saved at {i+1} questions.")

elapsed = time.time() - start_time
print(f"\nDone! {total} questions in {elapsed/60:.1f} minutes.")

Starting fresh inference.
Total questions: 643
Remaining: 643

  [10/643] 3.3 q/min | last: CORP-TAX-010
  [20/643] 3.3 q/min | last: CORP-TAX-020
  [30/643] 3.3 q/min | last: CORP-TAX-030
  [40/643] 3.3 q/min | last: CORP-TAX-040
  [50/643] 3.3 q/min | last: CORP-TAX-050
  [60/643] 3.3 q/min | last: CORP-TAX-060
  [70/643] 3.3 q/min | last: TAX-INTL-010
  [80/643] 3.3 q/min | last: TAX-INTL-020
  [90/643] 3.3 q/min | last: TAX-INTL-030
  [100/643] 3.3 q/min | last: TAX-INTL-040
  >> Checkpoint saved at 100 questions.
  [110/643] 3.3 q/min | last: TAX-INTL-050
  [120/643] 3.3 q/min | last: TAX-INTL-060
  [130/643] 3.3 q/min | last: TAX-INTL-070
  [140/643] 3.3 q/min | last: TAX-INTL-080
  [150/643] 3.2 q/min | last: ESTG-NEW-010
  [160/643] 3.2 q/min | last: ESTG-NEW-020
  [170/643] 3.2 q/min | last: VAT-DOM-010
  [180/643] 3.2 q/min | last: VAT-DOM-020
  [190/643] 3.2 q/min | last: VAT-DOM-030
  [200/643] 3.2 q/min | last: VAT-DOM-040
  >> Checkpoint saved at 200 questions.
  [210/643

## Cell 13 — Save Final Results
Converts the generated answers into the required submission format and writes them to a CSV file.

- `pd.DataFrame(results)` converts the Python result list into a pandas DataFrame.
- `result_df[["id", "answer"]]` keeps only the two required columns.
- `to_csv(OUTPUT_CSV, index=False)` writes the final result table to disk without an extra index column.

The results are already in the correct order because the inference loop iterates through the dataset sequentially. This cell formats and saves the output for submission.

### Project Impact
This cell turns model output into the official deliverable format of the course project. It ensures the RAG results can be submitted, compared, and scored without format-related problems.

In [21]:
# Convert results to DataFrame and save
result_df = pd.DataFrame(results)

# Keep only the required columns
result_df = result_df[["id", "answer"]]

# Save to CSV
result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Results saved to {OUTPUT_CSV}")
print(f"Shape: {result_df.shape}")
print(result_df.head())

Results saved to /kaggle/working/model3_rag_results.csv
Shape: (643, 2)
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Wenn eine Körperschaft ein zinsloses Darlehen ...
2  CORP-TAX-003  Gemäß § 13 Abs. 1 KStG müssen **Privatstiftung...
3  CORP-TAX-004  ##  Steuerliche Behandlung von Dividenden in Ö...
4  CORP-TAX-005  Die Unterscheidung zwischen offenen und verdec...


## Cell 14 — Verification
Validates whether the final CSV output is complete and correctly structured.

- `pd.read_csv(OUTPUT_CSV)` loads the saved result file back into memory.
- `expected = len(df)` stores how many rows the final output should contain.
- `assert len(result) == expected` checks that the number of generated answers matches the number of input questions.
- `assert list(result.columns) == ["id", "answer"]` checks that the output has exactly the required columns.
- `result["answer"].notna().all()` ensures that no answer cell is missing (NaN).
- `result["answer"].str.strip().ne("").all()` ensures that no answer is an empty string.
- `result["id"].is_unique` verifies that every question ID appears only once.

These assertions act as automatic sanity checks. If one of them fails, the notebook stops immediately and shows where the output format is wrong.

### Project Impact
This cell protects the project from submitting incomplete or invalid results. It ensures that the RAG output is not only generated, but also structurally correct and ready for evaluation.

In [22]:
result = pd.read_csv(OUTPUT_CSV)
expected = len(df)  # dynamic — avoids hardcoding row count
assert len(result) == expected, f"Expected {expected} rows, got {len(result)}"
assert list(result.columns) == ["id", "answer"], f"Wrong columns: {list(result.columns)}"
assert result["answer"].notna().all(), "Some answers are NaN"
assert result["answer"].str.strip().ne("").all(), "Some answers are empty strings"
assert result["id"].is_unique, "IDs are not unique"
print(f"Verification passed. {expected} answers ready.")
print(f"  Columns: {list(result.columns)}")
print(f"  Avg answer length: {result['answer'].str.len().mean():.0f} chars")

Verification passed. 643 answers ready.
  Columns: ['id', 'answer']
  Avg answer length: 780 chars


## Cell 15+16 — Statistics
Generating simple descriptive statistics and sample outputs for inspection.

- `merged.sample(5, random_state=42)` selects five random rows from the merged results table.
- `random_state=42` ensures that the same five rows are selected every time, which makes the sampling reproducible.
- `result["answer"].str.len()` calculates the length of each answer in characters.
- `answer_lengths.mean()` computes the average answer length.
- Similar statistics can also be shown for the median, minimum, and maximum answer length.

These statistics are useful for understanding the behavior of the model after generation. They can also help identify issues such as overly short answers, overly long answers, or inconsistent response patterns.

### Project Impact
This cell gives a quick overview of the output quality and consistency. It is especially useful for presentation, debugging, and comparing the behavior of the RAG model against the other project models.


In [24]:
# Load results and original dataset for comparison
result = pd.read_csv(OUTPUT_CSV)
df_orig = pd.read_csv(DATASET_PATH)

# Merge for side-by-side viewing
merged = df_orig.merge(result, on="id", how="inner")

# Show 5 random samples
print("=" * 80)
print("SAMPLE OUTPUTS (5 random questions)")
print("=" * 80)

samples = merged.sample(5, random_state=42)
for _, row in samples.iterrows():
    print(f"\n--- {row['id']} ---")
    print(f"Q: {row['prompt']}")
    print(f"A: {row['answer'][:500]}{'...' if len(str(row['answer'])) > 500 else ''}")
    print()

SAMPLE OUTPUTS (5 random questions)

--- SELF-070 ---
Q: Wann ist eine Sicherheitseinrichtung für Registrierkassen verpflichtend?
A: Laut **§ 131b EStG** ist eine Sicherheitseinrichtung für Registrierkassen **verbindlich**, wenn sie **in einem elektronischen System** verwendet wird.  


Es gibt aber auch **keine Pflicht**, eine Sicherheitsanlage für Registrierkassen zu installieren, wenn man **ein elektronisches Kassensystem** nutzt. 



Bitte beachten Sie, dass dies nur eine kurze Zusammenfassung ist. Es ist immer wichtig, sich bei komplexen Fragen an einen Steuerberater zu wenden.


--- VAT-INTL-047 ---
Q: "Die Firma hat ihren Hauptsitz in Graz und eine Filiale in Linz. Ein Buchhaltungsunternehmen aus Wien erledigt die Buchhaltung für die Filiale in Linz. Wo gilt die Leistung steuerlich als ausgeführt: am Sitz der Firma in Graz oder in Linz?"
A: Laut UStG § 1 Abs. 1 Z 1 und 2 ist die Ausübung von Gewerbe in Österreich nur dann mit einem Wohnsitz/Sitz/Betriebsstätte verbunden, wenn di

In [25]:
# Summary statistics on generated answers
answer_lengths = result["answer"].str.len()

print("Answer Length Statistics:")
print(f"  Mean:   {answer_lengths.mean():.0f} chars")
print(f"  Median: {answer_lengths.median():.0f} chars")
print(f"  Min:    {answer_lengths.min():.0f} chars")
print(f"  Max:    {answer_lengths.max():.0f} chars")
print(f"  Std:    {answer_lengths.std():.0f} chars")
print()

# Check for error answers
error_count = result["answer"].str.contains("Fehler bei der Generierung", na=False).sum()
print(f"Error answers: {error_count} / {len(result)}")
print(f"Success rate: {(len(result) - error_count) / len(result) * 100:.1f}%")

Answer Length Statistics:
  Mean:   780 chars
  Median: 763 chars
  Min:    155 chars
  Max:    1377 chars
  Std:    244 chars

Error answers: 0 / 643
Success rate: 100.0%


## Cell 17 — Cleanup
This cell frees memory after the notebook has finished running.

- `del model` removes the generator model variable from memory.
- `del tokenizer` removes the tokenizer object.
- `del retriever_model` removes the embedding model.
- `del index` deletes the FAISS search index.
- `del embeddings` deletes the stored chunk vectors.
- `gc.collect()` runs Python’s garbage collector to clean up unused objects.
- `torch.cuda.empty_cache()` releases cached GPU memory back to PyTorch.

This cleanup is especially useful in Kaggle or other notebook environments with limited resources. It helps avoid out-of-memory problems if additional cells are executed afterwards.

### Project Impact
Improves notebook stability after the main pipeline has finished. It is not part of answer generation itself, but it helps the project run more safely in constrained GPU environments.


In [26]:
# Free GPU memory
import gc

del model
del tokenizer
del retriever_model
del index
del embeddings
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

GPU memory freed.
